In [2]:
import pandas as pd

# Phase 1

# LOADING

In [3]:
kb = pd.read_json("C:\\Users\\ameys\\OneDrive\\Desktop\\CS\\PYTHON\\RAG_project\\multidoc2dial (2)\\multidoc2dial\\multidoc2dial_doc.json")
train_data = pd.read_json("C:\\Users\\ameys\\OneDrive\\Desktop\\CS\\PYTHON\\RAG_project\\multidoc2dial (2)\\multidoc2dial\\multidoc2dial_dial_train.json")

In [4]:
import re

def preprocess_text(txt):

    txt = txt.lower()

    txt = re.sub(r'\s+', ' ', txt)
    txt = txt.strip()

    return txt

In [5]:
knowledge_chunk = []

for domain,docs in kb['doc_data'].items():  #[ssa,va,dmv,sa] , [[...] , [...], [...], [...]]
    for doc_key,doc_content in docs.items():  #[[ssa[],....],....] , [[ssa[][]....],.....]

        spans_dict = doc_content.get('spans',{})
        for span_id,span_content in spans_dict.items():

            chunk = {
                "text" : span_content.get('text_sp').strip(),
                "metadata" : {
                    "domain" : domain,
                    "doc_id" : doc_content.get('doc_id'),
                    "title" : doc_content.get('title'),
                    "span_id" : span_id
                }
            }

            knowledge_chunk.append(chunk)

#1

## The code below has been used to create chunks

In [6]:
knowledge_chunk = []

for domain,docs in kb['doc_data'].items():  #[ssa,va,dmv,sa] , [[...] , [...], [...], [...]]
    for doc_key,doc_content in docs.items():  #[[ssa[],....],....] , [[ssa[][]....],.....]

        titlemain = doc_content.get('title','')
        spans_dict = doc_content.get('spans',{})

        for span_id,span_content in spans_dict.items():
            
            sectiontitle = span_content.get('title','')

            parent = span_content.get('parent_titles',[])
            parentlist = []

            for p in parent:

                if isinstance(p,dict) and 'text' in p:
                    parentlist.append(p['text'])

                elif isinstance(p,str):
                    parentlist.append(p)

            parts = " > ".join(parentlist)


            contextprefix = titlemain

            if(parts):
                contextprefix += f" > {parts}"

            if sectiontitle and sectiontitle != titlemain : contextprefix += f" > {sectiontitle}"
            # if parent: contextprefix += f" > {parent}"

            completetext = f"{span_content.get('text_sp').strip()} [SEP] {contextprefix}"
            # completetext = f"{contextprefix} [SEP] {span_content.get('text_sp').strip()}"

            chunk = {
                "text" : span_content.get('text_sp').strip(),
                "embedding_text" : completetext, #preprocess(completetext)
                "metadata" : {
                    "domain" : domain,
                    "doc_id" : doc_content.get('doc_id'),
                    "title" : doc_content.get('title'),
                    "span_id" : span_id
                }
            }

            knowledge_chunk.append(chunk)

In [ ]:
print(len(knowledge_chunk))

In [ ]:
import pprint

In [ ]:
print(knowledge_chunk[0])

## Code below is used for creating the list of question answers

In [7]:
train_qa = [] # -> list of question answer

#we will try to nest to the question and answer

for domain, doc_list in train_data['dial_data'].items():

    for doc in doc_list: #each doc_list are turns of convos
        dial_id = doc['dial_id']
        turns = doc['turns']

        for i,turn in enumerate(turns):

            if turn['role']=='agent':

                answer = turn['utterance']

                question = turns[i-1]['utterance'] if i > 0 else "" #we are assuming that convos always start with a question.
                for reference in turn.get('references',[]): #it is unlikely to be that question is ""

                    sample = {
                        "question" : question,
                        "answer" : answer,
                        "target_doc" : reference['doc_id'],
                        "target_span" : reference['id_sp'],
                        "domain" : domain
                    }

                    train_qa.append(sample)



In [9]:
print(len(train_qa))

39304


In [10]:
train_qa[0]

{'question': 'Hello, I forgot o update my address, can you help me with that?',
 'answer': 'hi, you have to report any change of address to DMV within 10 days after moving. You should do this both for the address associated with your license and all the addresses associated with all your vehicles.',
 'target_doc': 'Top 5 DMV Mistakes and How to Avoid Them#3_0',
 'target_span': '6',
 'domain': 'dmv'}

In [ ]:
train_qa[501]

Checking if the mapping is correct

In [ ]:
test_sample = train_qa[0]
print(test_sample) # testing if the question matches with the source text

source_text = next((item['text'] for item in knowledge_chunk if item['metadata']['span_id']==test_sample['target_span'] and item['metadata']['doc_id']==test_sample['target_doc']),"No match")

print(source_text)

##just checking if mapping is correct

In [8]:
import torch
import torch.nn.functional as F 
from transformers import AutoTokenizer, AutoModel
import numpy as np

In [9]:
import faiss

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [11]:
def mean_pooling(model_op, attention_mask):

    # print(model_op[0])
    token_embeds = model_op[0] # first element are all the token embeddings
    ip_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeds.size()).float()

    sum_embeddings = torch.sum(token_embeds * ip_mask_expanded, 1)
    sum_mask = torch.clamp(ip_mask_expanded.sum(1), min=1e-9)

    return sum_embeddings/sum_mask

# Loading

In [12]:
tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

model.to(device)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 384, padding_idx=0)
    (position_embeddings): Embedding(512, 384)
    (token_type_embeddings): Embedding(2, 384)
    (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-5): 6 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:

model.eval()

sentences = [f"{chunk['embedding_text']}" for chunk in knowledge_chunk] ## [chunk['text'] for chunk in knowledge_chunk]
batch_size = 64
# encodedips = tokenizer(sentences,padding=True,truncation=True,return_tensors='pt',max_length=256)
all_embeds = []
# with torch.no_grad():
#     model_output = model(**encodedips)

# sentence_Embeddings = mean_pooling(model_output,encodedips['attention_mask'])

# sentence_embeddings = F.normalize(sentence_Embeddings, p=2, dim=1)

# print(f"Embeddings Shape: {sentence_embeddings.shape}")

for i in range(0,len(sentences),batch_size):
    batch_text = sentences[i:i+batch_size]

    encodedips = tokenizer(
        batch_text,
        padding = True,
        truncation = True,
        return_tensors = 'pt',
        max_length = 256
    ).to(device)

    with torch.no_grad():
        model_op = model(**encodedips)


    sentence_Embeddings = mean_pooling(model_op,encodedips['attention_mask'])
    sentence_Embeddings = F.normalize(sentence_Embeddings, p=2, dim=1)

    all_embeds.append(sentence_Embeddings.cpu().numpy())

    if (i // batch_size) % 10 == 0:
        print(f"Processed {i + len(batch_text)}/{len(sentences)}...")

kb_embeddings = np.vstack(all_embeds)
print(f"Finished! Final embeddings shape: {kb_embeddings.shape}")


In [16]:
np.savez_compressed('rag_foundation.npz', 
                    embeddings=kb_embeddings, 
                    metadata=knowledge_chunk)

# for loading
# data = np.load('rag_foundation.npz', allow_pickle=True)
# kb_embeddings = data['embeddings']
# knowledge_chunk = data['metadata']

NameError: name 'kb_embeddings' is not defined

In [13]:
data = np.load('rag_foundation.npz', allow_pickle=True)
kb_embeddings = data['embeddings']
knowledge_chunk = data['metadata']

In [14]:
kb_embeddings.shape

(35659, 384)

In [15]:
dimension = kb_embeddings.shape[1]
embeddings_to_float32 = kb_embeddings.astype('float32')

index = faiss.IndexFlatL2(dimension)
index.add(embeddings_to_float32)

In [16]:
def pipeline(question):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
    model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

    model.to(device)
    model.eval()
    
    ques = [question]

    enc = tokenizer(
        ques,
        padding = True,
        truncation = True,
        return_tensors = 'pt',
        max_length = 256
    ).to(device)
    
    with torch.no_grad():
        model_output = model(**enc)

    ans = mean_pooling(model_output,enc['attention_mask'])
    ans = F.normalize(ans, p=2, dim=1)

    return ans.cpu().numpy().astype('float32')


In [21]:
k = 100
ques = pipeline(train_qa[0]['question'])
distance, indices = index.search(ques,k)
indices



c:\miniconda3\pkgs\rag_project_backup\lib\site-packages\transformers\models\bert\modeling_bert.py:413: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


array([[22836, 22799, 26328, 26324, 26327, 22821, 21492, 22815, 23666,
        22816, 22820, 22811, 20910, 21619, 22852, 23665, 21496, 21493,
        22827, 21620, 22803, 21497, 22822, 22851, 21495, 21499, 22800,
        17520,  5540, 22818,  5514, 22825, 18849, 22819, 21500, 22845,
        18850, 25902, 22804, 21618, 21616, 22829, 26326,  5551, 22846,
        22809,  5543, 22817, 18847, 26325, 21498,  5525, 22810, 22841,
        21604,  5517, 22813, 21605, 25824, 17244,  5553, 18000, 22849,
        22847, 21494,  5528,  5422,  5527, 22826,  5554, 17511, 25888,
        22801, 23675,  5310,  5512, 25891,  5538,  5544,  5518, 17516,
        22279,  5526, 22806, 22824, 22834, 25887,  5552, 24777, 18855,
        22814, 22805, 22832, 22839, 25900, 21592, 17527,  5545,  5519,
        22963]], dtype=int64)

In [22]:
train_qa[0]

{'question': 'Hello, I forgot o update my address, can you help me with that?',
 'answer': 'hi, you have to report any change of address to DMV within 10 days after moving. You should do this both for the address associated with your license and all the addresses associated with all your vehicles.',
 'target_doc': 'Top 5 DMV Mistakes and How to Avoid Them#3_0',
 'target_span': '6',
 'domain': 'dmv'}

In [ ]:
# for i,idx in enumerate(indices[0]):

#     match = knowledge_chunk[idx]
#     print(f"Rank {i+1} | Distance: {distance[0][i]:.4f}")
#     print(f"Document: {match['metadata']['doc_id']}")
#     print(f"Content: {match['text']}\n")

In [ ]:
# target_doc = train_qa[0]['target_doc']
# target_span = train_qa[0]['target_span']
# act_ind = -1

# for idx,chunk in enumerate(knowledge_chunk):
    
#     if chunk['metadata']['doc_id'] == target_doc and chunk['metadata']['span_id'] == target_span:
#         act_ind = i
#         break


In [ ]:
# print(act_ind)

In [ ]:
# if act_ind in indices[0] : 
#     rank = np.where(indices[0] == act_ind)[0][0] + 1
#     print(rank)
# else:
#     print("not found")

In [23]:
def filtered_search(question, target_domain, k = 500, kFinal = 10):
    q = pipeline(question)

    dist,ind = index.search(q,k)

    result = []

    for d,i in zip(dist[0],ind[0]):

        chunk = knowledge_chunk[i]

        if chunk['metadata']['domain'] == target_domain:
            result.append({
                "text" : chunk['text'],
                "doc_id" : chunk['metadata']['doc_id'],
                "distance" : d
            })

        if len(result) == kFinal:
            break

    return result

In [ ]:
train_qa[0]

In [ ]:
userques = train_qa[0]['question']
domain_user = train_qa[0]['domain']
import pprint
matchs = filtered_search(userques,domain_user)

for m in matchs:
    print(m)

In [24]:
fake_query = f"{train_qa[0]['domain']} [SEP] report any change of address to DMV within 10 days after moving"

q = pipeline(fake_query)

dist,ind = index.search(q,5)

In [25]:
for i,idx in enumerate(ind[0]):

    match = knowledge_chunk[idx]
    print(f"Rank {i+1} | Distance: {dist[0][i]:.4f}")
    print(f"Document: {match['metadata']['doc_id']}")
    print(f"Content: {match['text']}\n")

Rank 1 | Distance: 0.2540
Document: Top 5 DMV Mistakes and How to Avoid Them#3_0
Content: you must report a change of address to DMV within ten days of moving.

Rank 2 | Distance: 0.4472
Document: How to change your address#1_0
Content: you must report a change of residence address on your license, permit, non - driver ID or vehicle records to the DMV within 10 days.

Rank 3 | Distance: 0.4918
Document: Information about transaction entries#3_0
Content: New York State law requires you to report a change of residence to the DMV within 10 days.

Rank 4 | Distance: 0.5389
Document: ATVs: Information for Owners and Operators#3_0
Content: You also must report your address change to the DMV [6] within 10 days.

Rank 5 | Distance: 0.5818
Document: Information about transaction entries#3_0
Content: After your address change is processed ,



In [26]:
def metrics(qa_pairs,k=100):

    correct = 0
    total = len(qa_pairs)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
    model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2').to(device)
    model.eval()

    qs = [preprocess_text(q['question']) for q in qa_pairs]

    ip = tokenizer(qs,padding=True, truncation = True, max_length = 256, return_tensors = 'pt').to(device)

    with torch.no_grad():
        op = model(**ip)

    query_embeds = mean_pooling(op,ip['attention_mask'])
    query_embeds = F.normalize(query_embeds, p=2, dim=1).cpu().numpy().astype('float32')

    distances, indices = index.search(query_embeds,k=k)

    for i,sample in enumerate(qa_pairs):
        retrieved_metadata = [knowledge_chunk[idx]['metadata'] for idx in indices[i]]

        if any(sample['target_doc']==m['doc_id'] and sample['target_span']==m['span_id'] for m in retrieved_metadata):
            correct+=1

    print(f"Hit rate : {(correct/total)*100}")

metrics(train_qa[:100])

Hit rate : 50.0


# Phase 2

In [22]:
from sentence_transformers import CrossEncoder

In [23]:
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=device)

In [17]:
def rerankedMetrics(qa_pairs, k_r = 100, k_f = 100):

    correct = 0
    total = len(qa_pairs)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
    model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2').to(device)
    model.eval()

    qs = [preprocess_text(q['question']) for q in qa_pairs]

    ip = tokenizer(qs,padding=True, truncation = True, max_length = 256, return_tensors = 'pt').to(device)

    with torch.no_grad():
        op = model(**ip)

    query_embeds = mean_pooling(op,ip['attention_mask'])
    query_embeds = F.normalize(query_embeds, p=2, dim=1).cpu().numpy().astype('float32')

    distances, indices = index.search(query_embeds,k=k_r) # existing logic so far

    for i,sample in enumerate(qa_pairs):

        ques = sample['question']

        candidate_chunks = [knowledge_chunk[idx] for idx in indices[i]]

        pairs = [[ques,chunk['embedding_text']] for chunk in candidate_chunks] # pair question with every candidate document

        scores = reranker.predict(pairs)
        results = sorted(zip(candidate_chunks, scores), key=lambda x : x[1], reverse=True)

        top5Metadata = [c['metadata'] for c,score in results[:k_f]]

        if any(sample['target_doc'] == m['doc_id'] and sample['target_span'] == m['span_id'] 
               for m in top5Metadata):
            correct += 1

    print(f"Reranked hit rate : {(correct/total)*100}")

rerankedMetrics(train_qa[:100])
    

Reranked hit rate : 50.0


### Trying BM25 

In [18]:
from rank_bm25 import BM25Okapi

corpus = [chunk['embedding_text'].split(" ") for chunk in knowledge_chunk]
bm25 = BM25Okapi(corpus)


In [30]:
def bm25Test(qa_pairs,k=100):
    correct = 0
    total = len(qa_pairs)

    qs = [q['question'] for q in qa_pairs]
    ip = tokenizer(qs, padding=True, truncation=True, max_length=256, return_tensors='pt').to(device)

    with torch.no_grad():
        op = model(**ip)

    embeds = mean_pooling(op,ip['attention_mask'])
    embeds = F.normalize(embeds, p=2, dim=1).cpu().numpy().astype('float32')
    _,indices = index.search(embeds,k=k)

    for i,sample in enumerate(qa_pairs):

        vdb_hits = set(indices[i]) #faiss indexes

        tok_query = sample['question'].split(" ")
        bm25_scores = bm25.get_scores(tok_query)

        bm25_hits = set(np.argsort(bm25_scores)[::-1][:k]) # ?

        hybrid = vdb_hits | bm25_hits

        target_doc = sample['target_doc']
        target_span = sample['target_span']

        is_hit = any(
            knowledge_chunk[idx]['metadata']['doc_id']==target_doc and
            knowledge_chunk[idx]['metadata']['span_id']==target_span
            for idx in hybrid
        )

        if is_hit:
            correct+=1

    print(f"Hybrid hit rate : {(correct/total)*100}")

bm25Test(train_qa[:100])



Hybrid hit rate : 56.00000000000001


In [19]:
def bm25preprocess(text):

    return re.sub(r'[^\w\s]','',text).lower().split()

In [20]:
corpus = [bm25preprocess(chunk['embedding_text']) for chunk in knowledge_chunk]
bm25 = BM25Okapi(corpus)

In [25]:
def finalmetrics(qa_pairs, k_r = 100, k_f = 5):

    correct = 0
    total = len(qa_pairs)
    instructional_ques = [f"Represent this query for retrieving relevant documentation: {q['question']}" for q in qa_pairs]

    ip = tokenizer(instructional_ques, padding=True, truncation=True, max_length=256, return_tensors='pt').to(device)

    with torch.no_grad():
        op = model(**ip)

    embeds = mean_pooling(op,ip['attention_mask'])
    embeds = F.normalize(embeds, p=2, dim=1).cpu().numpy().astype('float32')
    _,indices = index.search(embeds,k=k_r)

    for i,sample in enumerate(qa_pairs):

        vdb_hits = set(indices[i]) #faiss indexes

        tok_query = bm25preprocess(sample['question'])
        bm25_scores = bm25.get_scores(tok_query)

        bm25_hits = set(np.argsort(bm25_scores)[::-1][:k_r]) # ?

        hybrid = list(vdb_hits | bm25_hits)
        candidate_chunks = [knowledge_chunk[idx] for idx in hybrid]

        pairs = [[sample['question'], c['embedding_text']] for c in candidate_chunks]

        scores = reranker.predict(pairs)

        results = sorted(zip(candidate_chunks, scores), key=lambda x: x[1], reverse=True)

        top_metadata = [c['metadata'] for c,scores in results[:k_f]]

        if any(sample['target_doc']==m['doc_id'] and sample['target_span']==m['span_id'] for m in top_metadata):
            correct+=1
        

    print(f"final hit rate : {(correct/total)*100}")

finalmetrics(train_qa[:100])

final hit rate : 23.0


In [24]:
def finalmetrics2(qa_pairs,k_r = 100, k_f = 5):

    correct = 0
    total = len(qa_pairs)

    qs = [q['question'] for q in qa_pairs]

    for j in range(0,total,100):
        ip = tokenizer(qs[j:j+100], padding=True, truncation=True, max_length=256, return_tensors='pt').to(device)

        with torch.no_grad():
            op = model(**ip)

        embeds = mean_pooling(op,ip['attention_mask'])
        embeds = F.normalize(embeds, p=2, dim=1).cpu().numpy().astype('float32')
        _,indices = index.search(embeds,k=k_r)

        for i,sample in enumerate(qa_pairs[j:j+100]):

            tgt_domain = sample['domain']

            vdb_hits = indices[i]
            vdb_filtered = [idx for idx in vdb_hits if knowledge_chunk[idx]['metadata']['domain'] == tgt_domain]

            tok_query = bm25preprocess(sample['question'])
            bm25_scores = bm25.get_scores(tok_query)

            bm25globalIndex = np.argsort(bm25_scores)[::-1][:200] #getting the top 200 global scores
            bm25_filtered = [idx for idx in bm25globalIndex if knowledge_chunk[idx]['metadata']['domain'] == tgt_domain]

            hybrid = list(set(vdb_filtered) | set(bm25_filtered))

            if not hybrid: #for no matches
                continue 
            
            candidate_chunks = [knowledge_chunk[idx] for idx in hybrid]

            pairs = [[sample['question'],c['embedding_text']] for c in candidate_chunks]

            scores = reranker.predict(pairs)
            results = sorted(zip(candidate_chunks, scores), key=lambda x: x[1], reverse=True)

            top_metadata = [c['metadata'] for c, score in results[:k_f]]

            if any(sample['target_doc']==m['doc_id'] and sample['target_span']==m['span_id'] for m in top_metadata):
                correct+=1

    print(f"Hit rate : {(correct/total)*100}")

finalmetrics2(train_qa[:100])

Hit rate : 25.0
